In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#modelling
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor 
import warnings


In [2]:
df=pd.read_csv("../datasets/student-mat.csv")
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [7]:
x = df.drop('G1', axis=1)

In [8]:
x.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,no,4,3,4,1,1,3,6,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,no,5,3,3,1,1,3,4,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,no,4,3,2,2,3,3,10,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,yes,3,2,2,1,1,5,2,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,no,4,3,2,1,2,5,4,10,10


In [9]:
y=df['G1']

In [10]:
print(y)

0       5
1       5
2       7
3      15
4       6
       ..
390     9
391    14
392    10
393    11
394     8
Name: G1, Length: 395, dtype: int64


#create column transformer with 3 types of transformer 

In [11]:
#creates column transformer with 3 types of transformer 
num_features =x.select_dtypes(exclude="object").columns
cat_features=x.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
oh_transformer=OneHotEncoder()

preprocessor =ColumnTransformer(
    [
        ("OneHotEncoder",oh_transformer,cat_features),
        ("StandardScaler",numeric_transformer,num_features),
    ]
)

C:\Users\MY DELL\AppData\Local\Temp\ipykernel_24020\2284856750.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features=x.select_dtypes(include="object").columns


In [12]:
print(cat_features)

Index(['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
       'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities',
       'nursery', 'higher', 'internet', 'romantic'],
      dtype='str')


In [13]:
print(num_features)

Index(['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G2', 'G3'],
      dtype='str')


In [14]:
x=preprocessor.fit_transform(x)

In [15]:
x


array([[ 1.        ,  0.        ,  1.        , ...,  0.03642446,
        -1.25479105, -0.96493392],
       [ 1.        ,  0.        ,  1.        , ..., -0.21379577,
        -1.52097927, -0.96493392],
       [ 1.        ,  0.        ,  1.        , ...,  0.53686493,
        -0.72241461, -0.0907392 ],
       ...,
       [ 0.        ,  1.        ,  0.        , ..., -0.33890588,
        -0.72241461, -0.74638524],
       [ 0.        ,  1.        ,  0.        , ..., -0.71423623,
         0.34233827, -0.0907392 ],
       [ 0.        ,  1.        ,  0.        , ..., -0.08868565,
        -0.45622639, -0.30928788]], shape=(395, 58))

In [16]:
x.shape

(395, 58)

In [18]:
#separate dataset into train and test 
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train.shape,x_test.shape

((316, 58), (79, 58))

In [1]:
#create an evaluate function to give all metrics after model training 
def evaluate_model(true,predicted):
    mae=mean_absolute_error(true,predicted)
    mse=mean_squared_error(true,predicted)
    rmse=np.sqrt(mean_squared_error(true,predicted))
    r2_square=r2_score(True,predicted)
    return mae,rmse,r2_square

In [ ]:
models={
    "Linear Regressor": LinearRegression(),
    "Lasso":Lasso(),
    "Ridge":Ridge(),
    "K-Neighbors Regressor":KNeighborsRegressor(),
    "Decision Tree":DecisionTreeRegressor(),
    "Random Forest Regressor":RandomForestRegressor(),
    "XGBRegressor":XGBRegressor(),
    "CatBoosting Regressor":CatBoostRegressor(verbose=False),
    "AdaBoost Regressor":AdaBoostRegressor()


}
model_list=[]
r2_list=[]

for i in range(len(list(models))):
    model=list(models.value())[i]
    model.fit(x_train,y_train)#train model

    #make prediction 

y_train_pred=model.predict(x_train)
y_test_pred=model.predict(x_test)

#evaluate Train and Test 
model_train_mae,model_train_rmse,model_train_r2=evaluate_model(y_train,y_train_pred)
model_test_mae,model_test_rmse,model_test_r2=evaluate_model(y_train,y_train_pred)
model_test_mae
print(list(models.keys())[i])
model_list.append(list(models.keys())[i])
print('Model performance for training set')
print("_Root  Mean squared error:{:.4f}.format(model(trainer_rmse))")
print("_root absolute error:{:.4f}".format(model_test_mae))
print("_R2 Score:{:4f}".format(model_train_mae))
print("-R2 score:{:4f}".format(model_train_r2))
print('----------------------')
print('model performance for test set')
print("-Root Mean squared error :{:4f}".format(model_test_rmse))
print("-mean absolute error:{:.4f}".format(model_test_mae))
print('-R2 score :{:4f}'.format(model_test_r2))
r2_list.append(model_test_r2)
print("="*35)
print("\n")

#RESULT 
pd.DataFrame(list(zip(model_list,r2_list))),columns=['Model Name ','R2_score']).sort_values(by=["R2_score"],ascending=False)

#linear Regression 

lin_model=LinearRegression(fit_intercept=True)
lin_model=lin_model.fit(x_train,y_train)
y_pred=lin_model.pred(x_test)
score=r2_score(y_test,y_pred)*100
print("Accuracy of the model is %2f"%score)
#plot y_pred and y_test
plt.scatter(y_test,y_pred);
plt.xlabel('actual')
plt.ylabel('Predicted')
sns.regplot(x=y_test,y=y_pred,ci=None,color='red');

#dif between actual and predicted value 
pred_df=pd.DataFrame({'actual value':y_test,'Predicted value':y_pred})
pred_df

